# 실습4 · BM25

직접 만든 역색인(bm25_tf · bm25_df)과 `bm25_search()` 로 BM25 를 재고, 트리거가 역색인을 맞추는 것을 확인한다.
강의 : 「BM25 개념」 · 「BM25 공식 해설」 · 「BM25 토큰 수와 커버리지」

| 셀 | 하는 일 |
|---|---|
| 1 | 설정 |
| 2 | 연결과 평가 도구 |
| 3 | 재료 — N · avgdl · 흔한 단어와 드문 단어의 IDF |
| 4 | 공식을 항 단위로 — 질문 하나, 정답 지문 하나 |
| 5 | k1(포화) · b(길이 보간) 곡선 |
| 6 | ts_rank 와 순위 비교 |
| 7 | 트리거 — 넣고 · 고치고 · 지우고 · 정합성 |
| 8 | 평가와 k1 · b 바꿔 보기 |
| 9 | 토큰 수와 IDF — IDF 를 끄면 |
| 10 | ts_rank 와 비교표 |

실습3 을 먼저 하면 셀 10 에서 ts_rank 결과와 나란히 본다.

### 셀 1 · 설정

In [ ]:
# 셀 1 · 설정
DB_URL = "postgresql://lab:lab@db:5432/lab"
EVAL_N = 500
K1, B  = 1.2, 0.75                                    # bm25_search 기본값
QUESTION, GOLD = "말라카이트에서 나온 색깔을 사용한 에디션은?", 1   # 셀 4·6 의 질문과 정답 지문 id

### 셀 2 · 연결과 평가 도구
`BM25_SQL` 은 `bm25_search()` 와 같은 식을 CTE 로 풀어 쓴 것. `{idf}` 자리를 바꿔 셀 9 에서 IDF 를 끈다.

In [ ]:
# 셀 2 · 연결과 평가 도구
import time
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import psycopg
from pgvector.psycopg import register_vector

plt.rcParams.update({"font.family": "NanumGothic", "axes.unicode_minus": False, "figure.dpi": 110})
pd.set_option("display.max_colwidth", 70)

conn = psycopg.connect(DB_URL, autocommit=True)
conn.prepare_threshold = None      # SET 으로 바꾼 설정이 매번 반영되게 (실행 계획 재사용 끔)
register_vector(conn)              # numpy 배열 ↔ vector 타입


def sql(query, params=None):
    """SELECT 결과를 표(DataFrame)로"""
    cur = conn.execute(query, params)
    return pd.DataFrame(cur.fetchall(), columns=[d.name for d in cur.description]) if cur.description else None


# 평가용 질문 : 11개마다 하나씩 고른 EVAL_N 개. 모든 노트북이 같은 질문으로 잰다
E = f"(SELECT id, doc_id AS gold, question, embedding, emb128 FROM questions WHERE id % 11 = 0 ORDER BY id LIMIT {EVAL_N}) e"
RESULTS = Path("results/검색평가.csv")


def evaluate(name, query, params=None):
    """query 는 (질문 id, 정답 지문 id, 상위 문서 id 배열) 을 질문마다 한 행씩 돌려주는 SQL.
    정답 지문이 상위 10위 안이면 Hit, MRR = 정답 순위 역수의 평균. 결과는 RESULTS 에 누적"""
    t = time.perf_counter()
    rows = conn.execute(query, params).fetchall()
    sec = time.perf_counter() - t
    rank = [list(ids[:10]).index(g) + 1 if g in ids[:10] else None for _, g, ids in rows]
    res = {"방법": name, "Hit@10": round(np.mean([r is not None for r in rank]), 3),
           "MRR": round(np.mean([1 / r if r else 0 for r in rank]), 3), "질문수": len(rows), "초": round(sec, 1)}
    RESULTS.parent.mkdir(exist_ok=True)
    new = pd.DataFrame([res])
    if RESULTS.exists():
        old = pd.read_csv(RESULTS)
        new = pd.concat([old[old["방법"] != name], new])
    new.to_csv(RESULTS, index=False)
    return res


# bm25_search() 와 같은 식. {q} 질의문 · {idf} IDF 식 · {k1} {b} 상수
BM25_SQL = """
  WITH s  AS (SELECT count(*)::float8 AS n, avg(doclen)::float8 AS avgdl FROM docs),
       qt AS (SELECT DISTINCT lexeme AS term FROM unnest(to_tsvector('korean', {q}))),
       w  AS (SELECT qt.term, {idf} AS idf FROM qt JOIN bm25_df d ON d.term = qt.term CROSS JOIN s)
  SELECT t.doc_id FROM w
  JOIN bm25_tf t ON t.term = w.term JOIN docs d ON d.id = t.doc_id CROSS JOIN s
  GROUP BY t.doc_id
  ORDER BY sum(w.idf * t.tf * ({k1} + 1) / (t.tf + {k1} * (1 - {b} + {b} * d.doclen / s.avgdl))) DESC, t.doc_id
  LIMIT 10"""
IDF_ON  = "ln(1 + (s.n - d.df + 0.5) / (d.df + 0.5))"
IDF_OFF = "1.0"
print("평가 질문", conn.execute(f"SELECT count(*) FROM {E}").fetchone()[0], "개")

### 셀 3 · 재료 — N · avgdl · IDF
- N · avgdl : 검색할 때마다 docs 에서 바로 계산
- 흔한 형태소는 IDF 가 0 에 가깝고, 한 지문에만 나오는 형태소가 가장 높다

In [ ]:
# 셀 3 · 재료
display(sql("SELECT count(*) AS N, round(avg(doclen), 1) AS avgdl, sum(doclen) AS 전체형태소 FROM docs"))
idf = "round(ln(1 + ((SELECT count(*) FROM docs) - df + 0.5) / (df + 0.5))::numeric, 3)"
display(sql(f"SELECT term AS 흔한형태소, df, {idf} AS idf FROM bm25_df ORDER BY df DESC LIMIT 8").T)
sql(f"""SELECT term AS 형태소, df, {idf} AS idf FROM bm25_df
        WHERE term IN ('말라카이트', '에디션', '색깔', '반도체', '사용', '것') ORDER BY df""").T

### 셀 4 · 공식을 항 단위로
정답 지문 하나에 대해 질의 형태소마다 tf · idf · 항 점수. 합계가 BM25 점수.
- 지문에 없는 형태소는 tf 0 → 항 0

In [ ]:
# 셀 4 · 공식을 항 단위로
T4 = sql("""
WITH s  AS (SELECT count(*)::float8 AS n, avg(doclen)::float8 AS avgdl FROM docs),
     qt AS (SELECT DISTINCT lexeme AS term FROM unnest(to_tsvector('korean', %(q)s)))
SELECT qt.term AS 형태소, df.df, round(ln(1 + (s.n - df.df + 0.5) / (df.df + 0.5))::numeric, 3) AS idf,
       coalesce(t.tf, 0) AS tf, d.doclen, round(s.avgdl::numeric, 1) AS avgdl,
       round((ln(1 + (s.n - df.df + 0.5) / (df.df + 0.5)) * coalesce(t.tf, 0) * (%(k1)s + 1)
              / (coalesce(t.tf, 0) + %(k1)s * (1 - %(b)s + %(b)s * d.doclen / s.avgdl)))::numeric, 3) AS 항점수
FROM qt CROSS JOIN s JOIN docs d ON d.id = %(gold)s
LEFT JOIN bm25_df df ON df.term = qt.term
LEFT JOIN bm25_tf t  ON t.term = qt.term AND t.doc_id = d.id
ORDER BY 항점수 DESC
""", {"q": QUESTION, "gold": GOLD, "k1": K1, "b": B})
display(T4)
print("합계 =", T4["항점수"].sum(), " ← bm25_search 점수:",
      round(conn.execute("SELECT score FROM bm25_search(%s, 50) WHERE doc_id = %s", (QUESTION, GOLD)).fetchone()[0], 3))

### 셀 5 · k1 · b 곡선
- 왼쪽 : 문서 길이 = 평균일 때 tf 항 = tf·(k1+1)/(tf+k1). 상한 k1+1 에서 포화
- 오른쪽 : tf = 3 고정, 문서 길이 / 평균 에 따라. b 가 클수록 긴 문서를 많이 깎음

In [ ]:
# 셀 5 · k1 · b 곡선
tf = np.arange(0, 31)
ratio = np.linspace(0.25, 3, 50)
fig, ax = plt.subplots(1, 2, figsize=(13, 3.8))
for k1, c in [(0.5, "#009CA6"), (1.2, "#3D484A"), (2.0, "#FF7043"), (5.0, "#9aa5a8")]:
    ax[0].plot(tf, tf * (k1 + 1) / (tf + k1), color=c, label=f"k1={k1}")
ax[0].set_xlabel("tf"); ax[0].set_ylabel("tf 항"); ax[0].legend(); ax[0].set_title("k1 : 포화의 세기")
for b, c in [(0.0, "#009CA6"), (0.75, "#3D484A"), (1.0, "#FF7043")]:
    ax[1].plot(ratio, 3 * 2.2 / (3 + 1.2 * (1 - b + b * ratio)), color=c, label=f"b={b}")
ax[1].set_xlabel("문서 길이 / 평균 길이"); ax[1].set_ylabel("tf=3 의 항"); ax[1].legend(); ax[1].set_title("b : 길이 보간의 세기")
plt.tight_layout(); plt.show()

### 셀 6 · ts_rank 와 순위 비교
같은 질문, 상위 5개. ★ = 정답 지문.

In [ ]:
# 셀 6 · ts_rank 와 순위 비교
bm = sql("SELECT r.doc_id AS id, round(r.score::numeric, 2) AS bm25, left(d.title, 38) AS 제목 "
         "FROM bm25_search(%s, 5) r JOIN docs d ON d.id = r.doc_id ORDER BY r.score DESC", (QUESTION,))
ts = sql("""SELECT id, round(ts_rank(tsv, tq)::numeric, 4) AS ts_rank, left(title, 38) AS 제목
            FROM docs, (SELECT replace(plainto_tsquery('korean', %s)::text, '&', '|')::tsquery AS tq) x
            WHERE tsv @@ tq ORDER BY ts_rank(tsv, tq) DESC, id LIMIT 5""", (QUESTION,))
for t in (bm, ts):
    t.insert(0, "정답", np.where(t["id"] == GOLD, "★", ""))
pd.concat({"BM25": bm, "ts_rank (OR)": ts}, axis=1)

### 셀 7 · 트리거 — 넣고 · 고치고 · 지우고
시험용 지문 2개를 넣어 역색인이 따라오는지 본다. 마지막에 지워서 원래대로.
- INSERT : 새 형태소 '큐비트' 가 bm25_df 에 생기고, '말라카이트' df 1 → 2
- UPDATE : 본문을 고칠 때는 tsv · doclen 도 같이 다시 넣는다 → 트리거가 역색인을 고침
- UPDATE(source 만) : tsv 가 그대로라 역색인은 안 건드림
- DELETE : 원래 값으로
- 정합성 : 트리거가 유지한 tf · df = docs.tsv 에서 처음부터 다시 센 값 → 불일치 0

In [ ]:
# 셀 7 · 트리거 실험
def df_of(*terms):
    got = dict(conn.execute("SELECT term, df FROM bm25_df WHERE term = ANY(%s)", (list(terms),)).fetchall())
    return {t: got.get(t, 0) for t in terms}

log = [("처음", df_of("큐비트", "말라카이트", "양자컴퓨터"))]
conn.execute("""
  INSERT INTO docs (doc_key, title, content, source, tsv, doclen)
  SELECT k, t, c, 'test', doc_tsv(t, c), tsv_len(doc_tsv(t, c))
  FROM (VALUES ('test-1', '양자컴퓨터 상용화 경쟁', '국내 연구진이 큐비트 100개 규모의 양자컴퓨터 시제품을 공개했다. 큐비트 오류율을 낮췄다.'),
               ('test-2', '말라카이트 원석 전시회', '녹색 광물 말라카이트 원석을 모은 전시회가 열린다.')) v(k, t, c)""")
log.append(("INSERT 2건", df_of("큐비트", "말라카이트", "양자컴퓨터")))
new_doc = sql("SELECT r.doc_id, round(r.score::numeric, 2) AS score, d.title FROM bm25_search('큐비트', 3) r JOIN docs d ON d.id = r.doc_id")

c2 = "국내 연구진이 양자컴퓨터 시제품을 공개했다."
conn.execute("""UPDATE docs SET content = %(c)s, tsv = doc_tsv(title, %(c)s), doclen = tsv_len(doc_tsv(title, %(c)s))
                WHERE doc_key = 'test-1'""", {"c": c2})
log.append(("UPDATE 본문", df_of("큐비트", "말라카이트", "양자컴퓨터")))
conn.execute("UPDATE docs SET source = 'test-updated' WHERE doc_key LIKE 'test-%'")
log.append(("UPDATE source", df_of("큐비트", "말라카이트", "양자컴퓨터")))
conn.execute("DELETE FROM docs WHERE doc_key LIKE 'test-%'")
log.append(("DELETE", df_of("큐비트", "말라카이트", "양자컴퓨터")))

display(pd.DataFrame({k: v for k, v in log}).T.rename_axis("df 변화"))
display(new_doc)
sql("""
SELECT 'tf 불일치' AS 검사, count(*) AS 건수 FROM (
  (SELECT term, doc_id, tf FROM bm25_tf EXCEPT SELECT t.lexeme, d.id, cardinality(t.positions) FROM docs d, unnest(d.tsv) t)
  UNION ALL
  (SELECT t.lexeme, d.id, cardinality(t.positions) FROM docs d, unnest(d.tsv) t EXCEPT SELECT term, doc_id, tf FROM bm25_tf)) x
UNION ALL
SELECT 'df 불일치', count(*) FROM (
  (SELECT term, df FROM bm25_df EXCEPT SELECT term, count(*)::int FROM bm25_tf GROUP BY term)
  UNION ALL
  (SELECT term, count(*)::int FROM bm25_tf GROUP BY term EXCEPT SELECT term, df FROM bm25_df)) x
UNION ALL
SELECT 'doclen 불일치', count(*) FROM docs WHERE doclen <> tsv_len(tsv)
""")

### 셀 8 · 평가와 k1 · b 바꿔 보기
기본값(k1 1.2 · b 0.75)으로 평가하고, k1 · b 를 바꿔 Hit@10 · MRR 이 어떻게 움직이는지 본다. 약 1분.

In [ ]:
# 셀 8 · 평가와 k1 · b
def bm25_eval(name, k1=K1, b=B, idf=IDF_ON):
    return evaluate(name, f"SELECT e.id, e.gold, ARRAY({BM25_SQL.format(q='e.question', idf=idf, k1=k1, b=b)}) FROM {E}")

base = bm25_eval("BM25")
grid = pd.DataFrame([{"k1": k1, "b": b, **bm25_eval(f"BM25 k1={k1} b={b}", k1, b)}
                     for k1, b in [(1.2, 0.0), (1.2, 0.5), (1.2, 1.0), (0.5, 0.75), (2.0, 0.75)]])
pd.concat([pd.DataFrame([{"k1": K1, "b": B, **base}]), grid])[["k1", "b", "Hit@10", "MRR", "초"]]

### 셀 9 · 토큰 수와 IDF — IDF 를 끄면
강의 : 질의 토큰이 많을수록 IDF 의 영향이 줄고 커버리지(몇 개를 담았나)가 순위를 정한다.
질문을 **질의 형태소 수**로 묶어, IDF 를 켠 BM25 와 끈 BM25(모든 단어 IDF = 1)의 차이를 본다.

In [ ]:
# 셀 9 · 토큰 수와 IDF
rows = conn.execute(f"""
  SELECT e.id, e.gold, (SELECT count(DISTINCT lexeme) FROM unnest(to_tsvector('korean', e.question))) AS n,
         ARRAY({BM25_SQL.format(q='e.question', idf=IDF_ON,  k1=K1, b=B)}) AS idf_on,
         ARRAY({BM25_SQL.format(q='e.question', idf=IDF_OFF, k1=K1, b=B)}) AS idf_off
  FROM {E}""").fetchall()
D = pd.DataFrame(rows, columns=["id", "gold", "n", "on", "off"])
rr = lambda ids, g: 1 / (list(ids).index(g) + 1) if g in ids else 0
D["MRR IDF 켬"] = [rr(a, g) for a, g in zip(D["on"], D["gold"])]
D["MRR IDF 끔"] = [rr(a, g) for a, g in zip(D["off"], D["gold"])]
D["형태소 수"] = pd.cut(D["n"], [0, 4, 6, 8, 30], labels=["4 이하", "5~6", "7~8", "9 이상"])
G = D.groupby("형태소 수", observed=True)[["MRR IDF 켬", "MRR IDF 끔"]].mean()
G["질문수"] = D.groupby("형태소 수", observed=True).size()
G["차이"] = G["MRR IDF 켬"] - G["MRR IDF 끔"]
ax = G[["MRR IDF 켬", "MRR IDF 끔"]].plot.bar(figsize=(9, 3.4), color=["#3D484A", "#FF7043"], rot=0)
ax.set_ylim(0, 1); ax.set_title("질의 형태소 수별 MRR"); plt.show()
G.round(3)

### 셀 10 · ts_rank 와 비교표
`results/검색평가.csv` 에 쌓인 결과 (실습3 을 했다면 ts_rank 계열도 함께).

In [ ]:
# 셀 10 · 비교표
R = pd.read_csv(RESULTS)
R = R[~R["방법"].str.startswith("BM25 k1=")].sort_values("MRR", ascending=False)
R